# SimpleEdgeNet Training — Mu2e Calorimeter GNN

This notebook runs the full pipeline on EAF:
1. Environment check (packages, GPU)
2. Build graphs from ROOT files (CPU) — if not already done
3. Train SimpleEdgeNet (GPU)
4. Evaluate and plot results

**Working directory:** `/exp/mu2e/app/users/wzhou2/projects/calorimeter/GNN/`

In [ ]:
import os
os.chdir("/exp/mu2e/app/users/wzhou2/projects/calorimeter/GNN")

import sys
sys.path.insert(0, ".")

## 1. Environment Check

In [ ]:
import torch
import torch_geometric
import uproot
import scipy
import yaml
import numpy as np

print(f"Python:          {sys.version}")
print(f"PyTorch:         {torch.__version__}")
print(f"PyG:             {torch_geometric.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")
    print(f"CUDA version:    {torch.version.cuda}")
else:
    print("WARNING: No GPU detected. Training will be slow on CPU.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {DEVICE}")

In [ ]:
# Load config
with open("configs/default.yaml") as f:
    cfg = yaml.safe_load(f)

print("Config loaded.")
print(f"  Model: {cfg['model']['name']}")
print(f"  Hidden dim: {cfg['model']['hidden_dim']}")
print(f"  MP layers: {cfg['model']['n_mp_layers']}")
print(f"  Epochs: {cfg['train']['epochs']}")
print(f"  Batch size: {cfg['train']['batch_size']}")
print(f"  Truth mode: {cfg['data']['truth_mode']}")

## 2. Build Graphs (if needed)

This step reads ROOT files from `/pnfs/` and saves `.pt` graph files.
Files are on **tape** — first access may be slow (staging).

Skip this cell if `data/processed/` already has enough graphs.

In [ ]:
from pathlib import Path

processed_dir = Path(cfg["data"]["processed_dir"])
existing_graphs = sorted(processed_dir.glob("*.pt"))
print(f"Existing processed graphs: {len(existing_graphs)}")

# Check how many are in each split
from src.data.dataset import CaloGraphDataset

def load_split(path):
    with open(path) as f:
        return [l.strip() for l in f if l.strip()]

train_files = load_split(cfg["data"]["splits"]["train"])
val_files = load_split(cfg["data"]["splits"]["val"])
test_files = load_split(cfg["data"]["splits"]["test"])

ds_train = CaloGraphDataset(processed_dir, file_list=train_files)
ds_val = CaloGraphDataset(processed_dir, file_list=val_files)
ds_test = CaloGraphDataset(processed_dir, file_list=test_files)

print(f"  Train: {len(ds_train)} graphs (from {len(train_files)} ROOT files)")
print(f"  Val:   {len(ds_val)} graphs (from {len(val_files)} ROOT files)")
print(f"  Test:  {len(ds_test)} graphs (from {len(test_files)} ROOT files)")

In [ ]:
# ============================================================
# BUILD GRAPHS — adjust n_files to control how many to process
# Set SKIP_BUILD = True if you already have enough graphs
# ============================================================
SKIP_BUILD = False
N_TRAIN_FILES = 10   # number of train ROOT files to process (max 35)
N_VAL_FILES = 4      # number of val ROOT files to process (max 7)
N_EVENTS = None      # None = all events per file, or set e.g. 100 for quick test

In [ ]:
if not SKIP_BUILD:
    import time
    from src.data.dataset import extract_events_from_file
    from src.geometry.crystal_geometry import load_crystal_map

    crystal_map = load_crystal_map(cfg["data"]["crystal_geometry"])
    graph_cfg = cfg["graph"]
    truth_mode = cfg["data"].get("truth_mode", "bfs_pseudo")
    processed_dir.mkdir(parents=True, exist_ok=True)

    def build_split(file_list, n_files, split_name):
        files = file_list[:n_files]
        print(f"\nBuilding {split_name}: {len(files)} files")
        total = 0
        t0 = time.time()
        for fi, fp in enumerate(files):
            fname = Path(fp).stem
            print(f"  [{fi+1}/{len(files)}] {Path(fp).name}...", end=" ", flush=True)
            ft0 = time.time()
            n = 0
            for data, ev_idx, disk_id, diag in extract_events_from_file(
                fp, crystal_map, graph_cfg,
                truth_mode=truth_mode, max_events=N_EVENTS,
            ):
                out_path = processed_dir / f"{fname}_evt{ev_idx:06d}_disk{disk_id}.pt"
                torch.save(data, out_path)
                n += 1
            total += n
            print(f"{n} graphs ({time.time()-ft0:.1f}s)")
        print(f"  {split_name} total: {total} graphs in {time.time()-t0:.1f}s")
        return total

    build_split(train_files, N_TRAIN_FILES, "train")
    build_split(val_files, N_VAL_FILES, "val")
    print("\nDone building graphs.")
else:
    print("Skipping graph build.")

In [ ]:
# Recompute normalization stats from train split
if not SKIP_BUILD:
    from src.data.normalization import compute_normalization_stats, save_stats

    ds_train_raw = CaloGraphDataset(processed_dir, file_list=train_files)
    print(f"Computing normalization stats from {len(ds_train_raw)} train graphs...")
    stats = compute_normalization_stats(ds_train_raw)
    save_stats(stats, cfg["data"]["normalization_stats"])
else:
    print("Using existing normalization stats.")

## 3. Load Datasets

In [ ]:
from src.data.normalization import load_stats, normalize_graph

stats = load_stats(cfg["data"]["normalization_stats"])

def norm_transform(data):
    return normalize_graph(data, stats)

train_dataset = CaloGraphDataset(processed_dir, file_list=train_files, transform=norm_transform)
val_dataset = CaloGraphDataset(processed_dir, file_list=val_files, transform=norm_transform)

print(f"Train: {len(train_dataset)} graphs")
print(f"Val:   {len(val_dataset)} graphs")

if len(val_dataset) == 0:
    print("WARNING: Val set is empty. Go back and build val graphs.")

In [ ]:
# Class balance
from src.training.losses import compute_class_weights

train_raw = CaloGraphDataset(processed_dir, file_list=train_files)
cw = compute_class_weights(train_raw)
pos_weight = cw["pos_weight"]

print(f"Positive edges: {cw['n_pos']} ({100*cw['n_pos']/(cw['n_pos']+cw['n_neg']):.1f}%)")
print(f"Negative edges: {cw['n_neg']} ({100*cw['n_neg']/(cw['n_pos']+cw['n_neg']):.1f}%)")
print(f"pos_weight (neg/pos): {pos_weight.item():.4f}")

## 4. Train

In [ ]:
from datetime import datetime
from src.models.simple_edge_net import SimpleEdgeNet
from src.training.trainer import Trainer

model = SimpleEdgeNet(
    node_dim=6, edge_dim=8,
    hidden_dim=cfg["model"]["hidden_dim"],
    n_mp_layers=cfg["model"]["n_mp_layers"],
    dropout=0.1,
)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"SimpleEdgeNet: {n_params:,} parameters")
print(model)

In [ ]:
# Training config — adjust as needed
train_cfg = dict(cfg["train"])
train_cfg["epochs"] = 100
train_cfg["batch_size"] = 32
train_cfg["early_stop_patience"] = 15

run_name = datetime.now().strftime("%Y%m%d_%H%M%S") + "_simple_edge_net"
run_dir = Path(cfg["output"]["run_dir"]) / run_name

trainer = Trainer(
    model=model,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    cfg=train_cfg,
    pos_weight=pos_weight,
    device=DEVICE,
    run_dir=run_dir,
)

print(f"Run dir: {run_dir}")
print(f"Device: {DEVICE}")

In [ ]:
# Save config + metadata
import json, subprocess

run_dir.mkdir(parents=True, exist_ok=True)
with open(run_dir / "config.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

try:
    git_hash = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"],
                                        stderr=subprocess.DEVNULL).decode().strip()
except Exception:
    git_hash = "unknown"

with open(run_dir / "metadata.json", "w") as f:
    json.dump({"git_hash": git_hash, "device": str(DEVICE),
               "timestamp": datetime.now().isoformat(),
               "n_train": len(train_dataset), "n_val": len(val_dataset)}, f, indent=2)

print(f"Metadata saved. git={git_hash}")

In [ ]:
# === TRAIN ===
history = trainer.fit()

## 5. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = [h["epoch"] for h in history]
train_loss = [h["train"]["loss"] for h in history]
val_loss = [h["val"]["loss"] for h in history]
train_f1 = [h["train"]["f1"] for h in history]
val_f1 = [h["val"]["f1"] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(epochs, train_loss, label="train")
axes[0].plot(epochs, val_loss, label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss")
axes[0].legend()

# F1
axes[1].plot(epochs, train_f1, label="train")
axes[1].plot(epochs, val_f1, label="val")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1")
axes[1].set_title("Edge F1 (positive class)")
axes[1].legend()

# Val precision/recall
val_p = [h["val"]["precision"] for h in history]
val_r = [h["val"]["recall"] for h in history]
axes[2].plot(epochs, val_p, label="precision")
axes[2].plot(epochs, val_r, label="recall")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Score")
axes[2].set_title("Val Precision / Recall")
axes[2].legend()

plt.tight_layout()
plt.savefig(run_dir / "training_curves.png", dpi=150)
plt.show()
print(f"Saved to {run_dir / 'training_curves.png'}")

In [ ]:
# Val AUC curves
if any(h["val"].get("roc_auc", 0) > 0 for h in history):
    val_roc = [h["val"].get("roc_auc", 0) for h in history]
    val_pr = [h["val"].get("pr_auc", 0) for h in history]

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(epochs, val_roc, label="ROC AUC")
    ax.plot(epochs, val_pr, label="PR AUC")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("AUC")
    ax.set_title("Val AUC")
    ax.legend()
    plt.tight_layout()
    plt.savefig(run_dir / "val_auc.png", dpi=150)
    plt.show()
else:
    print("No val AUC data (val set may be empty).")

## 6. Evaluate Best Model

In [ ]:
from torch_geometric.loader import DataLoader
from src.training.metrics import edge_metrics, edge_auc, cluster_metrics_from_edges

# Load best checkpoint
ckpt_path = run_dir / "checkpoints" / "best_model.pt"
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded best model from epoch {ckpt['epoch']} (val F1={ckpt['val_f1']:.4f})")
else:
    print("No checkpoint found — using final model state.")

model.to(DEVICE)
model.eval()
print("Model ready for evaluation.")

In [ ]:
# Evaluate on val set
if len(val_dataset) > 0:
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    all_logits, all_targets, all_masks = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            logits = model(batch)
            all_logits.append(logits.cpu())
            all_targets.append(batch.y_edge.cpu())
            all_masks.append(batch.edge_mask.cpu())

    logits_cat = torch.cat(all_logits)
    targets_cat = torch.cat(all_targets)
    masks_cat = torch.cat(all_masks)

    em = edge_metrics(logits_cat, targets_cat, masks_cat)
    auc = edge_auc(logits_cat, targets_cat, masks_cat)

    print("=== Val Edge Metrics ===")
    print(f"  Precision: {em['precision']:.4f}")
    print(f"  Recall:    {em['recall']:.4f}")
    print(f"  F1:        {em['f1']:.4f}")
    print(f"  Accuracy:  {em['accuracy']:.4f}")
    print(f"  ROC AUC:   {auc['roc_auc']:.4f}")
    print(f"  PR AUC:    {auc['pr_auc']:.4f}")
    print(f"  Positive edges: {em['n_pos']}, Negative edges: {em['n_neg']}")
else:
    print("Val set empty — skipping evaluation.")

In [ ]:
# Per-graph cluster metrics on val
if len(val_dataset) > 0:
    val_loader_single = DataLoader(val_dataset, batch_size=1, shuffle=False)
    purities, completenesses = [], []

    with torch.no_grad():
        for data in val_loader_single:
            data = data.to(DEVICE)
            logits = model(data)
            probs = torch.sigmoid(logits).cpu()
            cm = cluster_metrics_from_edges(
                data.edge_index.cpu(), probs, data.edge_mask.cpu(),
                data.hit_truth_cluster.cpu(), data.x.shape[0],
            )
            if cm["n_truth_clusters"] > 0:
                purities.append(cm["purity"])
                completenesses.append(cm["completeness"])

    if purities:
        print("=== Val Cluster Metrics ===")
        print(f"  Mean purity:       {np.mean(purities):.4f} +/- {np.std(purities):.4f}")
        print(f"  Mean completeness: {np.mean(completenesses):.4f} +/- {np.std(completenesses):.4f}")
        print(f"  Graphs evaluated:  {len(purities)}")
    else:
        print("No graphs with truth clusters found.")
else:
    print("Val set empty — skipping cluster metrics.")

## 7. Summary

In [ ]:
print(f"Run directory: {run_dir}")
print(f"Best val F1: {trainer.best_val_f1:.4f}")
print(f"Total epochs: {len(history)}")
print(f"\nFiles in run dir:")
for f in sorted(run_dir.rglob("*")):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.relative_to(run_dir)} ({size_kb:.0f} KB)")